# Strategy 16 — Deep Reinforcement Learning Trading Agent

> **Status: PLAN ONLY — no code cells yet.** This notebook is the human-readable
> review surface for the implementation plan. Each section below describes
> what the eventual code cell will do, what it depends on, and which design
> decisions are still open.
>
> See [[../DocumentationVault/strategies/16_Deep_RL_Trading]] for the strategy
> specification (Obsidian).

---

## §0. Prerequisites and environment

This notebook trains deep RL policies and **requires CUDA**. Before running:

- **GPU:** NVIDIA RTX 3060 Laptop (6 GB) — already present on the dev box.
- **Driver:** ≥ 560 (CUDA 12.6 runtime). `nvidia-smi` should report a CUDA
  version of 12.x or 13.x.
- **Python:** the repo's `.venv` runs Python 3.14.4. PyTorch's stable cu126
  wheels may not yet publish 3.14 builds at the time of writing — see the
  install instructions below for the nightly / Python-3.12 fallback paths.

### Install commands (run **outside** the notebook, in the shell)

```bash
# 1. PyTorch CUDA build — stable wheel from PyTorch's own index
source .venv/bin/activate.fish
pip install --upgrade pip
pip install --index-url https://download.pytorch.org/whl/cu126 torch torchvision

# 2. If Python 3.14 stable wheels aren't published yet, try nightly:
#    pip install --pre --index-url https://download.pytorch.org/whl/nightly/cu126 torch torchvision

# 3. RL stack
pip install "stable-baselines3[extra]>=2.3" "gymnasium>=0.29"

# 4. Verify
python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"
```

### Data dependency

The notebook reads multi-timeframe parquet caches built by `source.spark_loader`.
**That module is currently only on the `feature/multi-filter-portfolio-system`
branch** — it must be cherry-picked into this branch (or that PR merged into
`main`) before the data cells run. See the strategy doc's *Implementation Notes*
section for the rationale.

JDK 17 or 21 is required for PySpark 4.x. `get_spark()` auto-detects; if the
only JDK on this box is 26, install one of the supported versions:

```bash
sudo pacman -S jdk21-openjdk    # Arch / CachyOS
```

### Reference links

- PyTorch install selector: <https://pytorch.org/get-started/locally/>
- stable-baselines3 docs: <https://stable-baselines3.readthedocs.io/>
- Gymnasium docs: <https://gymnasium.farama.org/>


In [ ]:
# §0 — Environment preflight. Prints what is installed; does NOT install
# anything (the shell commands in the markdown above do that).
import importlib, shutil, sys, platform

print("Python  :", sys.version.split()[0], "—", platform.platform())
print("Java    :", shutil.which("java") or "NOT FOUND — install JDK 17 or 21 for PySpark")

try:
    import torch
    cuda_ok = torch.cuda.is_available()
    msg = f"PyTorch : {torch.__version__}  CUDA={cuda_ok}"
    if cuda_ok:
        msg += f"  device={torch.cuda.get_device_name(0)}"
    print(msg)
    if not cuda_ok:
        print("  [!] CPU-only — training will be infeasibly slow; debug runs only.")
except ImportError:
    print("PyTorch : NOT INSTALLED — see install commands above (§0)")

for mod in ("stable_baselines3", "gymnasium"):
    try:
        m = importlib.import_module(mod)
        print(f"{mod:8s}: {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"{mod:8s}: NOT INSTALLED — see install commands above (§0)")

try:
    import pyspark
    print("PySpark :", pyspark.__version__)
except ImportError:
    print("PySpark : NOT INSTALLED — pip install -r ../requirements.txt")


## §1. Imports

The eventual code cell will import:

- **Stdlib:** `gc`, `os`, `sys`, `pathlib.Path`, `dataclasses`, `random`.
- **PyData:** `numpy`, `pandas`, `matplotlib.pyplot`.
- **PyTorch:** `torch` — also asserts `torch.cuda.is_available()` and prints
  `torch.cuda.get_device_name(0)`. If CUDA is unavailable the cell prints a
  loud warning and continues with `device="cpu"` (debug only — training will
  be infeasibly slow).
- **RL stack:** `gymnasium as gym`, `stable_baselines3` (`PPO`, `DQN`),
  `stable_baselines3.common.callbacks` (`EvalCallback`, `CheckpointCallback`),
  `stable_baselines3.common.vec_env` (`DummyVecEnv`, `SubprocVecEnv`),
  `stable_baselines3.common.monitor.Monitor`.
- **Repo `source` imports:**
    - `source.spark_loader` — `get_spark`, `build_spark_grid` (parquet
      multi-TF cache; see §0 dependency note).
    - `source.backtest.Backtester`, `source.metrics.compute_metrics`,
      `source.dashboard.plot_backtest_dashboard`.
    - `source.robustness` — `block_bootstrap_trades`, `subperiod_analysis`,
      `parameter_sensitivity`, `monte_carlo_trades`.
    - `source.comparison.STRATEGY_REGISTRY` (read-only, for benchmark
      comparison in §13).
    - `source.parallel.parallel_map` (only used in §13 if running benchmarks
      in parallel).
- **New (to be added under `source/rl/`):**
    - `source.rl.env.TradingEnv`
    - `source.rl.train` — `train_one_seed`, `latest_checkpoint`,
      `evaluate_policy_to_signals`, `make_vec_env`.
- **Random seeds** pinned globally: `random.seed`, `np.random.seed`,
  `torch.manual_seed`, `torch.cuda.manual_seed_all` — all set from
  `GLOBAL_SEED = 42`. SB3 seeds are derived per training run.


In [ ]:
# §1 — Imports + global seeds.
import gc, os, random, sys, warnings
from dataclasses import asdict, replace
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from source import (
    DeepRLTradingStrategy, DeepRLTradingParams,
    Backtester, compute_metrics, plot_backtest_dashboard,
    walk_forward, default_score,
    block_bootstrap_trades, subperiod_analysis, monte_carlo_trades,
    STRATEGY_REGISTRY, make_params_for_group,
    cpu_count, parallel_map,
    get_spark, build_spark_grid, PYSPARK_AVAILABLE,
)

# New RL helpers (source/rl/)
from source.rl.env import (
    TradingEnv, compute_feature_panel, FEATURE_COLUMNS, ACTION_TO_SIGNAL,
)
from source.rl.train import (
    build_model, checkpoint_prefix, cuda_is_available,
    evaluate_policy_to_signals, latest_checkpoint, make_vec_env, train_one_seed,
)

warnings.filterwarnings("ignore")

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
try:
    import torch
    torch.manual_seed(GLOBAL_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(GLOBAL_SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print(f"GLOBAL_SEED={GLOBAL_SEED}  DEVICE={DEVICE}  cpus={cpu_count()}  "
      f"PySpark={PYSPARK_AVAILABLE}")


## §2. Configuration

Single dict cell pinning every knob so the rest of the notebook reads as
"apply config to data, env, agent, evaluator". Mirrors the layout of
`technical_analysis/15_multi_filter_portfolio_system.ipynb` §2.

### Markets and timeframes

```python
GROUP_TIMEFRAMES = {
    "forex": ["1h", "4h"],          # 1D dropped — too few bars for RL
    "b3":    ["30min", "1h"],
    # "crypto": [...]               # skipped — no data/crypto/
}
ASSETS = {
    "forex": ["EURUSD", "EURCAD", "GBPCHF"],
    "b3":    ["WDO", "WIN"],
}
WFO_ASSETS = {                       # hyperparameter selection happens on these
    "forex": ["EURUSD"],
    "b3":    ["WIN"],
}
```

### Train / validation / OOS split

```python
SPLITS = {
    "forex": {"train": ("2016-01-01", "2022-12-31"),
              "val":   ("2023-01-01", "2023-12-31"),
              "oos":   ("2024-01-01", "2026-12-31")},
    "b3":    {"train": ("2021-01-01", "2023-12-31"),
              "val":   ("2024-01-01", "2024-09-30"),
              "oos":   ("2024-10-01", "2026-12-31")},
}
```

### RL hyperparameters

```python
RL_CONFIG = dict(
    algorithms          = ["PPO", "DQN"],
    n_seeds             = 3,
    total_timesteps     = 1_000_000,
    n_envs              = 8,
    eval_freq           = 50_000,
    save_freq           = 100_000,
    obs_window          = 32,
    episode_len         = 2048,
    gamma               = 0.99,
    tx_cost_bps_by_grp  = {"forex": 5, "b3": 10},
    policy_arch         = (64, 64),
    device              = "cuda",
)
```

### Disabled-feature flags (per repo convention)

```python
DISABLED_V1 = dict(
    use_continuous_action       = False,
    use_differential_sharpe     = False,
    use_drawdown_penalty        = False,
    use_holding_penalty         = False,
    use_lstm_policy             = False,
    use_cnn_policy              = False,
    use_flat_close_signal       = False,
)
```

Open question for review: should `WFO_ASSETS` ever include a *second*
representative asset per group to guard against single-asset hyperparameter
overfit? The trade-off is roughly 2× training cost.


In [ ]:
# §2 — Configuration. Edit here, the rest of the notebook reads from these.
GROUP_TIMEFRAMES = {
    "forex": ["1h", "4h"],
    "b3":    ["30min", "1h"],
}
ASSETS = {
    "forex": ["EURUSD", "EURCAD", "GBPCHF"],
    "b3":    ["WDO", "WIN"],
}
WFO_ASSETS = {  # hyperparameter search subset (§9.1)
    "forex": ["EURUSD"],
    "b3":    ["WIN"],
}
SPLITS = {
    "forex": {"train": ("2016-01-01", "2022-12-31"),
              "val":   ("2023-01-01", "2023-12-31"),
              "oos":   ("2024-01-01", "2026-12-31")},
    "b3":    {"train": ("2021-01-01", "2023-12-31"),
              "val":   ("2024-01-01", "2024-09-30"),
              "oos":   ("2024-10-01", "2026-12-31")},
}

RL_CONFIG = dict(
    algorithms          = ["PPO", "DQN"],
    n_seeds             = 3,
    total_timesteps     = 1_000_000,
    n_envs              = 8,
    eval_freq           = 50_000,
    save_freq           = 100_000,    # CheckpointCallback writes every save_freq env-steps
    obs_window          = 32,
    episode_len         = 2048,
    gamma               = 0.99,
    learning_rate       = 3e-4,
    tx_cost_bps_by_grp  = {"forex": 5.0, "b3": 10.0},
    policy_arch         = (64, 64),
    device              = DEVICE,
)

DISABLED_V1 = dict(
    use_continuous_action       = False,
    use_differential_sharpe     = False,
    use_drawdown_penalty        = False,
    use_holding_penalty         = False,
    use_lstm_policy             = False,
    use_cnn_policy              = False,
    use_flat_close_signal       = False,
)

# --- runtime gates -------------------------------------------------------
# RUN_TRAINING        : actually launch the §9 training loop (resumes if checkpoints exist).
# RUN_HP_GRID         : run the §9.1 PPO/DQN hyperparameter scan (heavy; opt-in).
# SUBPROC_VECENV      : True → SubprocVecEnv (slower startup, real parallelism on Linux);
#                       False → DummyVecEnv (single process, easier to debug).
RUN_TRAINING   = True
RUN_HP_GRID    = False
SUBPROC_VECENV = False

CACHE_DIR  = REPO_ROOT / "data" / "_spark_cache"
MODEL_DIR  = REPO_ROOT / "machine_learning" / "_models_16_deep_rl"
TB_LOG_DIR = MODEL_DIR / "tb"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
TB_LOG_DIR.mkdir(parents=True, exist_ok=True)

n_cells = sum(len(ASSETS[g]) * len(tfs) for g, tfs in GROUP_TIMEFRAMES.items())
n_runs  = len(RL_CONFIG["algorithms"]) * RL_CONFIG["n_seeds"] * n_cells
print(f"Markets       : {list(GROUP_TIMEFRAMES)}")
print(f"Timeframes    : {GROUP_TIMEFRAMES}")
print(f"WFO subset    : {WFO_ASSETS}")
print(f"Total cells   : {n_cells}   ·   total training runs (algos × seeds × cells) = {n_runs}")
print(f"MODEL_DIR     : {MODEL_DIR}")
print(f"RUN_TRAINING  = {RUN_TRAINING}   ·   RUN_HP_GRID = {RUN_HP_GRID}   ·   SUBPROC_VECENV = {SUBPROC_VECENV}")


## §3. Data — PySpark multi-timeframe load

Reuses the `source.spark_loader` module from the multi-filter system. The
goal here is the same as in notebook 15: never hold every CSV in RAM, fan out
worker processes that each open a small parquet slice.

The cell will:

1. `get_spark(java_home=...)` — instantiates a Spark session pinned to a
   compatible JDK (17 or 21). Errors loudly if only JDK 24+ is found.
2. `build_spark_grid(...)` — for every M1 source CSV, tumbles into the target
   timeframes (`1h`, `4h`, `30min`) and writes parquet keyed by source mtime.
   This is a one-time cost per data update.
3. `read_parquet_slice(group, tf, asset, start, end)` — returns a small
   `pd.DataFrame` for the requested chronological slice (train / val / oos).
   Used by the env and by the evaluation pipeline.
4. Sanity prints: per `(group, tf, asset)` cell, bar count and date range for
   each split.

Per the parallelism convention (see CLAUDE memory): the parent process never
holds more than one DataFrame at a time — each training worker opens its own
parquet slice inside the worker process.

**Dependency reminder:** `source/spark_loader.py` is not on `main` yet —
either cherry-pick from `feature/multi-filter-portfolio-system` or wait for
that PR to merge. See §0.


In [ ]:
# §3 — Spark multi-timeframe parquet cache (one-time per data update; re-runs hit the cache).
spark = None
spark_grid: dict = {}
if PYSPARK_AVAILABLE:
    spark = get_spark()
    spark_grid = build_spark_grid(
        REPO_ROOT / "data", GROUP_TIMEFRAMES,
        cache_dir=CACHE_DIR, asset_filter=ASSETS,
        spark=spark, progress=True,
    )
    print("\nResampled grid (cells = group × tf × asset):")
    for g, tfs in spark_grid.items():
        for tf, assets in tfs.items():
            print(f"  {g:6s} {tf:6s}  assets={sorted(assets)}")
else:
    print("PySpark unavailable — install per §0 then re-run from here.")


def slice_by_split(df: pd.DataFrame, group: str, split: str) -> pd.DataFrame:
    """Chronological slice of one per-cell DataFrame for train/val/oos."""
    start, end = SPLITS[group][split]
    mask = (df.index >= pd.Timestamp(start)) & (df.index < pd.Timestamp(end))
    return df.loc[mask].copy()


def iter_cells(asset_map: dict[str, list[str]] | None = None):
    """Yield (group, tf, asset, dataset) for cells present in spark_grid ∩ asset_map."""
    am = asset_map if asset_map is not None else ASSETS
    for g, tfs in spark_grid.items():
        for tf, assets in tfs.items():
            for asset, ds in assets.items():
                if asset in am.get(g, []):
                    yield g, tf, asset, ds


rows = []
for g, tf, asset, ds in iter_cells():
    with ds.using() as df:
        rows.append({
            "group": g, "tf": tf, "asset": asset,
            "bars": len(df),
            "train_bars": len(slice_by_split(df, g, "train")),
            "val_bars":   len(slice_by_split(df, g, "val")),
            "oos_bars":   len(slice_by_split(df, g, "oos")),
            "first": df.index.min().date(),
            "last":  df.index.max().date(),
        })
sanity_df = pd.DataFrame(rows)
display(sanity_df)


## §4. Data cleaning and preprocessing

For each `(group, tf, asset)` slice loaded by §3:

- **Drop rows with NaN OHLCV** (rare for the source data, but defensive).
- **Assert monotonic index** — `df.index.is_monotonic_increasing`. Raise if
  not (the Spark loader should already guarantee this).
- **Drop session-boundary partials** (B3 only) — first/last bar of each session
  often has stub volume. Drop where `tick_vol == 0` or `volume_ratio < 0.05`.
- **Chronological train/val/OOS split** — slice the frame into three
  non-overlapping windows by date. Print bar counts per split per cell.
- **Online normalisation only** — no global `StandardScaler.fit(train_df)`
  that leaks distribution-level info into the env. All z-scoring happens
  inside `TradingEnv` against a rolling lookback (default 252 bars).

No global feature table is materialised — that would defeat the lazy/parallel
design. Each env recomputes features from its own slice inside the worker
process.


In [ ]:
# §4 — Cleaning + per-split sanity (no global feature table is materialised).
def clean_split(df: pd.DataFrame, group: str) -> pd.DataFrame:
    """Drop NaN OHLCV, assert monotonic, drop B3 session-stub bars."""
    out = df.dropna(subset=["open", "high", "low", "close"]).copy()
    if not out.index.is_monotonic_increasing:
        out = out.sort_index()
    if group == "b3" and "tick_vol" in out.columns:
        sma = out["tick_vol"].rolling(20, min_periods=1).mean()
        ratio = out["tick_vol"] / sma.replace(0.0, np.nan)
        keep = (out["tick_vol"] > 0) & (ratio.fillna(1.0) >= 0.05)
        out = out.loc[keep]
    assert out.index.is_monotonic_increasing, "non-monotonic index after cleaning"
    return out


rows = []
for g, tf, asset, ds in iter_cells():
    with ds.using() as df:
        clean = clean_split(df, g)
        for sp in ("train", "val", "oos"):
            s = slice_by_split(clean, g, sp)
            rows.append({"group": g, "tf": tf, "asset": asset, "split": sp,
                         "bars": len(s),
                         "first": (s.index.min() if not s.empty else None),
                         "last":  (s.index.max() if not s.empty else None)})
display(
    pd.DataFrame(rows)
      .pivot_table(index=["group", "tf", "asset"], columns="split", values="bars")
      .reindex(columns=["train", "val", "oos"])
)


## §5. Feature engineering and technical indicators

The set of features the env exposes to the policy (also defined in the
strategy doc):

| Feature | Formula | Notes |
|---|---|---|
| `log_return` | `log(close_t / close_{t-1})` | 1-bar log return |
| `realized_vol` | rolling std of `log_return` over `rv_window=20` | |
| `rsi / 100` | Wilder RSI period 14, scaled to `[0, 1]` | |
| `atr_rel` | ATR-14 / close | unit-free volatility |
| `volume_ratio` | `tick_vol / SMA(tick_vol, vol_period=20)` | |
| `bb_pos` | `(close − bb_mid) / (bb_upper − bb_mid)` | ∈ `[−1, +1]` band position |
| `current_position` | `{−1, 0, +1}` carried from previous step's action | |
| `bars_in_position` | step count since last position change, normalised by `episode_len` | |

Implementation:

- All windowed features are computed via pandas `rolling(...)` once per
  episode (or once per env instantiation, then sliced per step) — *not*
  per-step, which would be O(n²).
- After the rolling features are computed, the env z-scores each column
  against a rolling `rolling_z_window=252`-bar lookback. The z-score uses
  *only* bars up to and including `t` (no look-ahead). The first
  `rolling_z_window` bars of each episode are skipped on `reset()`.
- The observation passed to the policy at step `t` is a flattened
  `obs_window × n_features` window plus the 2 position-state scalars.

**Open question for review:** is `obs_window = 32` too long (more parameters,
slower training) or too short (policy can't see weekly seasonality)?
Candidate WFO range `{16, 32, 64}`.


In [ ]:
# §5 — Preview the per-bar feature panel on one (group, tf, asset) cell.
#       The env recomputes these per-episode in the worker process — no global
#       feature table is built here.
def params_for_cell(group: str) -> DeepRLTradingParams:
    """Per-(group) DeepRLTradingParams: tx cost + B3 session window applied."""
    return DeepRLTradingParams(
        obs_window=RL_CONFIG["obs_window"],
        episode_len=RL_CONFIG["episode_len"],
        tx_cost_bps=RL_CONFIG["tx_cost_bps_by_grp"][group],
        gamma=RL_CONFIG["gamma"],
        learning_rate=RL_CONFIG["learning_rate"],
        policy_arch=RL_CONFIG["policy_arch"],
        session_start=(9 if group == "b3" else None),
        session_end=(18 if group == "b3" else None),
        **DISABLED_V1,
    )


preview_g, preview_tf, preview_asset = "forex", "1h", "EURUSD"
preview_params = params_for_cell(preview_g)
preview_ds = (
    spark_grid.get(preview_g, {}).get(preview_tf, {}).get(preview_asset)
    if spark_grid else None
)
if preview_ds is not None:
    with preview_ds.using() as df:
        clean = clean_split(df, preview_g)
        train = slice_by_split(clean, preview_g, "train")
        panel = compute_feature_panel(train, preview_params).dropna()
    print(f"{preview_g}/{preview_tf}/{preview_asset} train={len(train)} bars  "
          f"feature panel: {panel.shape}  features={FEATURE_COLUMNS}")
    display(panel.tail(3))
    display(panel[FEATURE_COLUMNS].describe().T)

    # Plot the (un-z-scored) feature panel for a recent window.
    view = panel.tail(800)
    fig, ax = plt.subplots(len(FEATURE_COLUMNS), 1, figsize=(13, 9), sharex=True)
    for a_, col in zip(ax, FEATURE_COLUMNS):
        a_.plot(view.index, view[col], lw=.6); a_.set_ylabel(col); a_.grid(alpha=.3)
    fig.suptitle(f"Raw feature panel — {preview_g}/{preview_tf}/{preview_asset}")
    fig.tight_layout(); plt.show()
else:
    print(f"{preview_g}/{preview_tf}/{preview_asset} not in spark_grid — adjust §2 config.")


## §6. Environment — `TradingEnv(gymnasium.Env)`

New module: `source/rl/env.py`. The env is the core RL abstraction — all the
trading semantics live here, not in the agent. Code-cell-to-be:

```python
class TradingEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, df, params: DeepRLTradingParams, mode: str = "train"):
        # mode ∈ {"train", "eval"}; train picks random slices, eval is deterministic full pass
        ...

    def reset(self, seed=None, options=None):
        # 1. Pick episode start (random for train, 0 for eval).
        # 2. Reset position, equity, bars_in_position.
        # 3. Warm-up rolling stats over first rolling_z_window bars.
        # 4. Return (obs, info).
        ...

    def step(self, action):
        # 1. Decode action ∈ {0, 1, 2} → target_position ∈ {−1, 0, +1}.
        # 2. Apply B3 session mask (force flat outside session).
        # 3. Compute reward = position_{t-1} · log_return_t − tx_cost · |Δposition|.
        # 4. Update position, advance index, recompute observation.
        # 5. Terminate on end-of-window or drawdown > max_drawdown_fraction.
        # 6. Return (obs, reward, terminated, truncated, info).
        ...

    @property
    def observation_space(self):
        # Box(shape=(obs_window * n_features + 2,))
        ...

    @property
    def action_space(self):
        # Discrete(3)
        ...
```

Subtleties to document in the code:

- **Reward timing.** The reward at bar `t` is earned by the position held
  *into* bar `t` (i.e. set by the action at `t-1`). The very first action of
  an episode contributes zero reward at step 0; reward begins at step 1.
- **Transaction cost** is charged on the bar where the action differs from
  the previous position, not on entry / exit specifically. This is a more
  honest representation for a continuous-action policy and a non-issue for
  discrete.
- **Drawdown circuit-breaker** terminates the episode (not a real-money
  stop-loss) — it prevents the policy from learning that "ride it out
  forever" is a valid strategy.
- **Vectorisation.** SB3 wraps with `Monitor` + `DummyVecEnv` / `SubprocVecEnv`.
  Use `SubprocVecEnv(n_envs=8)` for training (multi-process); `DummyVecEnv`
  for evaluation (single-process determinism).

**Open question for review:** termination on drawdown > 30 % — what fraction
of training episodes are expected to be terminated by this? If it's near
zero the circuit breaker is harmless; if it's high the policy may learn an
over-cautious profile. Worth a sanity print in the eventual code cell.


In [ ]:
# §6 — TradingEnv smoke test: build env on the preview cell and roll one
#       random-policy episode end-to-end. Validates obs shape, reward sign,
#       and the drawdown circuit-breaker is reachable but rare.
if preview_ds is not None:
    with preview_ds.using() as df:
        clean = clean_split(df, preview_g)
        train = slice_by_split(clean, preview_g, "train")
    env = TradingEnv(train, preview_params, mode="train")
    obs, _ = env.reset(seed=GLOBAL_SEED)
    print(f"obs           : shape={obs.shape}  dtype={obs.dtype}")
    print(f"action_space  : {env.action_space}")
    print(f"observation_space.shape: {env.observation_space.shape}")
    print(f"valid_start={env.valid_start}  n_bars={env._n}  episode_len={preview_params.episode_len}")

    rng = np.random.default_rng(GLOBAL_SEED)
    cum_reward, position_changes, terminated_flag = 0.0, 0, False
    last_pos = 0
    while True:
        a = int(rng.integers(0, 3))
        obs, r, terminated, truncated, info = env.step(a)
        cum_reward += r
        if info["position"] != last_pos:
            position_changes += 1
            last_pos = info["position"]
        if terminated or truncated:
            terminated_flag = bool(terminated)
            break
    print(f"\nrandom-policy episode → cum_reward={cum_reward:+.4f}  "
          f"position_changes={position_changes}  drawdown={info['drawdown']:.3f}  "
          f"terminated_by_DD={terminated_flag}")


## §7. Model definition — PPO and DQN

Both algorithms instantiated identically except for class:

```python
def make_model(algo: str, env, params: DeepRLTradingParams, log_dir: Path):
    common = dict(
        policy        = "MlpPolicy",
        env           = env,
        verbose       = 0,
        device        = params.device,
        tensorboard_log = str(log_dir),
        policy_kwargs = dict(net_arch=list(params.policy_arch)),
        seed          = params.training_seed,
    )
    if algo == "PPO":
        return PPO(**common,
                   learning_rate = 3e-4,
                   n_steps       = 2048,
                   batch_size    = 64,
                   n_epochs      = 10,
                   gamma         = params.gamma,
                   gae_lambda    = 0.95,
                   clip_range    = 0.2,
                   ent_coef      = 0.01,
                   vf_coef       = 0.5)
    elif algo == "DQN":
        return DQN(**common,
                   learning_rate           = 1e-4,
                   buffer_size             = 100_000,
                   learning_starts         = 10_000,
                   batch_size              = 64,
                   tau                     = 1.0,
                   gamma                   = params.gamma,
                   train_freq              = 4,
                   target_update_interval  = 1000,
                   exploration_fraction    = 0.1,
                   exploration_final_eps   = 0.05)
    raise ValueError(algo)
```

Architecture choices and their justifications are tabulated in the strategy
doc — this section's markdown summary will link to it rather than duplicate.


In [ ]:
# §7 — PPO + DQN factory. ``source.rl.train.build_model`` wraps SB3 with the
#       doc defaults (n_steps=2048, batch=64, etc.). Confirm the policy plumbing
#       on the preview cell without committing to any training.
if preview_ds is not None and (cuda_is_available() or DEVICE == "cpu"):
    with preview_ds.using() as df:
        clean = clean_split(df, preview_g)
        train = slice_by_split(clean, preview_g, "train")

    for algo in RL_CONFIG["algorithms"]:
        env = make_vec_env(train, preview_params, n_envs=1,
                           mode="train", seed=GLOBAL_SEED, subproc=False)
        model = build_model(algo, env, preview_params,
                            seed=GLOBAL_SEED, device=DEVICE,
                            tensorboard_log=str(TB_LOG_DIR))
        net = model.policy if algo == "PPO" else model.q_net
        n_params = sum(p.numel() for p in net.parameters())
        print(f"{algo:3s}  policy={type(model.policy).__name__:20s}  "
              f"n_params={n_params:,}  device={model.device}")
        env.close(); del model
    gc.collect()
else:
    print("Skipping model factory check — install torch+sb3+gymnasium (§0) first.")


## §8. Resume from previous training (checkpoint discovery)

This section is the core of "training is interruptible". The code cell will:

1. **Discover checkpoints** under `models/16_deep_rl/`.
2. For each `(algo, group, tf, asset, seed)` cell, find the latest
   `step<N>.zip` and (for DQN) its replay buffer pickle.
3. Build a tidy DataFrame:

   ```
   algo  group  tf     asset    seed  step_done  has_best
   PPO   forex  4h     EURUSD   0     400_000    True
   PPO   forex  4h     EURUSD   1     —          False
   ...
   ```

4. **Print a resume plan** — for each cell, either "resume from step N" or
   "start from scratch". The user reads this *before* §9 kicks off training
   to confirm nothing unexpected is being overwritten.

Helper to be added in `source/rl/train.py`:

```python
def latest_checkpoint(model_dir: Path, algo: str, group: str, tf: str,
                       asset: str, seed: int) -> tuple[Path | None, int]:
    ...  # returns (path_or_None, steps_already_trained)
```

**Why this matters:** RL training is the most fragile / expensive step in
the notebook. Treating checkpoints as first-class (and human-visible in §8
before training starts) is the cheapest insurance against "I trained for
2 hours and lost it".


In [ ]:
# §8 — Checkpoint discovery. Runs before §9 so you can see exactly which cells
#       will start fresh, which will resume, and which are already done.
#       ``latest_checkpoint`` is the single source of truth — same helper §9 uses.
def cell_seeds(asset_map: dict[str, list[str]] | None = None):
    """Yield (algo, group, tf, asset, seed) for every training cell in the grid."""
    am = asset_map if asset_map is not None else ASSETS
    for algo in RL_CONFIG["algorithms"]:
        for g, tfs in GROUP_TIMEFRAMES.items():
            for tf in tfs:
                for asset in am.get(g, []):
                    if asset not in spark_grid.get(g, {}).get(tf, {}):
                        continue
                    for seed in range(RL_CONFIG["n_seeds"]):
                        yield algo, g, tf, asset, seed


def resume_plan(asset_map: dict[str, list[str]] | None = None) -> pd.DataFrame:
    """Row-per-cell table: where each (algo, group, tf, asset, seed) left off."""
    target = RL_CONFIG["total_timesteps"]
    rows = []
    for algo, g, tf, asset, seed in cell_seeds(asset_map):
        _, step = latest_checkpoint(MODEL_DIR, algo, g, tf, asset, seed)
        best = MODEL_DIR / f"{checkpoint_prefix(algo, g, tf, asset, seed)}_best" / "best_model.zip"
        rows.append({
            "algo": algo, "group": g, "tf": tf, "asset": asset, "seed": seed,
            "step_done": step, "remaining": max(target - step, 0),
            "has_best": best.exists(),
            "action": ("skip — already at target" if step >= target
                       else ("resume" if step > 0 else "start from scratch")),
        })
    return pd.DataFrame(rows)


plan_df = resume_plan()
display(plan_df)
n_skip   = int((plan_df["action"] == "skip — already at target").sum())
n_resume = int((plan_df["action"] == "resume").sum())
n_fresh  = int((plan_df["action"] == "start from scratch").sum())
print(f"\nResume plan: {n_skip} already-done · {n_resume} will resume · {n_fresh} fresh starts")
print(f"Models live under: {MODEL_DIR}")
print("Review the table above — §9 will act on it. Nothing trains until you run §9.")


## §9. Model training

Per-cell training loop. The outer iteration is over the
`{PPO, DQN} × WFO_ASSETS × n_seeds` grid for hyperparameter selection
(§9.1), then over the full `ASSETS × n_seeds` grid for the chosen
hyperparameters (§9.2). Training is fanned out across processes via
`parallel_map` only at the *seed* level — multiple algorithms or assets
training in parallel would oversubscribe the GPU.

### 9.1 Hyperparameter selection on `WFO_ASSETS`

For each cell in `{PPO, DQN} × WFO_ASSETS`:

- Build a small grid: `gamma × learning_rate × policy_arch`
  (3 × 3 × 3 = 27 combos).
- Train `n_seeds = 3` per combo (81 training runs per cell).
- For each run:
    - Construct `TradingEnv` on the **train** split.
    - Construct a separate `TradingEnv` on the **val** split for
      `EvalCallback`.
    - Train for `total_timesteps`, saving via `CheckpointCallback` every
      `save_freq` steps and the best-by-val-reward via `EvalCallback`.
    - Save final + best to `models/16_deep_rl/...`.
- Rank by mean validation Sharpe across seeds; pick the best combo per
  `(algo, group)` cell. Store in `BEST_HP[(algo, group)]`.

### 9.2 Generalisation pass — full asset grid

For each cell in `{PPO, DQN} × ASSETS \ WFO_ASSETS`:

- Use `BEST_HP[(algo, group)]` from §9.1.
- Train `n_seeds = 3` runs on each asset's train split.
- Save best-by-val and final checkpoints.

### Training observability

- `TensorBoard` logs to `models/16_deep_rl/tb/` (gitignored).
- A summary cell at the bottom of §9 prints, per cell:
  best validation reward, training time, GPU memory peak.

**Open question for review:** the §9.1 grid is 81 runs × ~30 min = ~40 GPU
hours for forex alone. Is the time budget acceptable? Alternatives:
- Use Optuna for ~20 trials instead of grid (probably better).
- Reduce `total_timesteps` to 500k for the grid pass, 1M only for the
  generalisation pass.
- Drop `policy_arch` from the grid (saves 3×).


In [ ]:
# §9.1 — Hyperparameter scan on WFO_ASSETS (opt-in: RUN_HP_GRID).
#       A reduced grid (gamma × learning_rate × policy_arch = 2×2×2 = 8 combos
#       × n_seeds = 24 runs per (algo, group, tf) cell). Each call goes through
#       train_one_seed, so periodic save + resume from latest_checkpoint apply
#       here too — re-running this cell after a crash picks up where it left off.
#       Per-combo total_timesteps is HP_TOTAL_TIMESTEPS (defaults to 200k for
#       the scan; the §9.2 generalisation pass uses the full 1M).
import json
from itertools import product

HP_GRID = {
    "gamma":        [0.95, 0.99],
    "learning_rate":[1e-4, 3e-4],
    "policy_arch":  [(64, 64), (128, 128)],
}
HP_TOTAL_TIMESTEPS = 200_000
HP_DIR = MODEL_DIR / "hp_grid"
HP_DIR.mkdir(parents=True, exist_ok=True)
BEST_HP_PATH = HP_DIR / "best_hp.json"


def _hp_prefix(algo, g, tf, asset, combo_idx):
    return f"hp{combo_idx:02d}_{checkpoint_prefix(algo, g, tf, asset, 0)}"


hp_rows = []
if RUN_HP_GRID and RUN_TRAINING:
    target = HP_TOTAL_TIMESTEPS
    combos = list(product(*HP_GRID.values()))
    print(f"HP grid: {len(combos)} combos × {RL_CONFIG['n_seeds']} seeds "
          f"× algos × WFO cells = many runs. Cumulative resume per combo.")
    for algo in RL_CONFIG["algorithms"]:
        for g, tfs in GROUP_TIMEFRAMES.items():
            for tf in tfs:
                for asset in WFO_ASSETS.get(g, []):
                    if asset not in spark_grid.get(g, {}).get(tf, {}):
                        continue
                    tr_df, val_df = _build_train_val(g, tf, asset)
                    for ci, vals in enumerate(combos):
                        combo = dict(zip(HP_GRID.keys(), vals))
                        cell_dir = HP_DIR / _hp_prefix(algo, g, tf, asset, ci)
                        cell_dir.mkdir(parents=True, exist_ok=True)
                        for seed in range(RL_CONFIG["n_seeds"]):
                            _, step_done = latest_checkpoint(
                                cell_dir, algo, g, tf, asset, seed)
                            if step_done >= target:
                                status = "skip"
                            else:
                                status = ("resume" if step_done > 0 else "fresh")
                            print(f"[hp{ci:02d}] {algo} {g}/{tf}/{asset} "
                                  f"seed={seed} {combo} → {status}")
                            if status == "skip":
                                hp_rows.append({
                                    "algo": algo, "group": g, "tf": tf,
                                    "asset": asset, "seed": seed, "combo_idx": ci,
                                    **combo, "step_done": step_done, "status": "skip"})
                                continue
                            override = replace(
                                params_for_cell(g),
                                gamma=combo["gamma"],
                                learning_rate=combo["learning_rate"],
                                policy_arch=combo["policy_arch"],
                            )
                            info = train_one_seed(
                                tr_df, val_df, override,
                                algo=algo, group=g, tf=tf, asset=asset, seed=seed,
                                model_dir=cell_dir,
                                total_timesteps=target,
                                n_envs=RL_CONFIG["n_envs"],
                                eval_freq=RL_CONFIG["eval_freq"],
                                save_freq=RL_CONFIG["save_freq"],
                                device=RL_CONFIG["device"],
                                tensorboard_log=str(TB_LOG_DIR / "hp"),
                                subproc=SUBPROC_VECENV,
                            )
                            hp_rows.append({
                                "algo": algo, "group": g, "tf": tf,
                                "asset": asset, "seed": seed, "combo_idx": ci,
                                **combo,
                                "step_done": info["steps_done"],
                                "status": ("resumed" if info["resumed"] else "fresh")})
                            gc.collect()
                    del tr_df, val_df
                    gc.collect()
    print("\nHP grid done. Best-by-mean-val-Sharpe selection deferred to a "
          "follow-up cell (run §11 on the hp_grid checkpoints to score them).")
else:
    print("RUN_HP_GRID=False (§2). Default hyperparameters from RL_CONFIG are "
          "used for the §9.2 generalisation pass — flip the flag to scan.")

hp_summary_df = pd.DataFrame(hp_rows)
display(hp_summary_df)


In [ ]:
# §9.2 — Generalisation training pass: (algo × group × tf × asset × seed).
#       Each call to train_one_seed:
#         1. checks latest_checkpoint(model_dir, …) for an existing run;
#         2. resumes from it if found (continuing global step counter), else
#            starts fresh;
#         3. trains with CheckpointCallback writing every save_freq env-steps
#            (so a crash never costs more than one save_freq window);
#         4. saves the best-by-validation policy under <prefix>_best/.
#       So "save periodically" + "check existing previous training before
#       starting a new section" is enforced *per cell*, not just at the top.

def _build_train_val(group: str, tf: str, asset: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    ds = spark_grid[group][tf][asset]
    df = clean_split(ds.load(), group)
    ds.unload()
    return slice_by_split(df, group, "train"), slice_by_split(df, group, "val")


train_summary = []
if RUN_TRAINING:
    plan = resume_plan()
    target = RL_CONFIG["total_timesteps"]
    for _, row in plan.iterrows():
        algo, g, tf, asset = row["algo"], row["group"], row["tf"], row["asset"]
        seed = int(row["seed"])
        prefix = checkpoint_prefix(algo, g, tf, asset, seed)
        _, step_done = latest_checkpoint(MODEL_DIR, algo, g, tf, asset, seed)

        if step_done >= target:
            print(f"[skip ] {prefix}  step={step_done:>9,}  (already at target {target:,})")
            train_summary.append(dict(prefix=prefix, step_done=step_done,
                                      status="skip"))
            continue

        msg = (f"resume from step {step_done:,}" if step_done > 0
               else "fresh start (no prior checkpoint)")
        print(f"[train] {prefix}  {msg}  →  target {target:,}")

        tr_df, val_df = _build_train_val(g, tf, asset)
        info = train_one_seed(
            tr_df, val_df, params_for_cell(g),
            algo=algo, group=g, tf=tf, asset=asset, seed=seed,
            model_dir=MODEL_DIR,
            total_timesteps=target,
            n_envs=RL_CONFIG["n_envs"],
            eval_freq=RL_CONFIG["eval_freq"],
            save_freq=RL_CONFIG["save_freq"],
            device=RL_CONFIG["device"],
            tensorboard_log=str(TB_LOG_DIR),
            subproc=SUBPROC_VECENV,
        )
        train_summary.append(dict(
            prefix=prefix,
            step_done=info["steps_done"],
            resumed=info["resumed"],
            status=("resumed" if info["resumed"] else "fresh"),
        ))
        del tr_df, val_df
        gc.collect()
    print("\nDone — see ``train_summary`` for per-cell outcomes.")
else:
    print("RUN_TRAINING=False (§2). Flip the flag and re-run to (re)train; "
          "§8's resume plan tells you exactly what would happen.")

train_summary_df = pd.DataFrame(train_summary)
display(train_summary_df)


## §10. WFO of the trained policy — temporal generalisation only

Re-uses `source.wfo.walk_forward` in a non-standard way: instead of WFO-ing
*strategy hyperparameters*, we WFO-evaluate the **same trained policy**
across consecutive OOS chunks. The grid has a single combo (the trained
policy), and the folds carve the OOS window into 3 chronological slices.

This answers a specific question: *does the policy's edge decay over the
OOS window, or is it stationary?* A canonical hyperparameter WFO is
intentionally out of scope for the RL training itself (see §9.1 reasoning).

The cell will:

1. For each `(algo, group, tf, asset)` cell with a best-checkpoint from §9.
2. Load the best-by-validation checkpoint.
3. `evaluate_policy_to_signals(model, oos_df, deterministic=True)` →
   pre-computed signal array.
4. Wrap with `DeepRLTradingStrategy` (signal-replay) and feed to
   `Backtester.run(...)`.
5. Run `walk_forward(...)` with `n_folds=3` over OOS to get fold-level
   metrics.
6. Plot `plot_wfo_dashboard(...)` per cell — equity curves and metric
   evolution across folds.


In [ ]:
# §10 — OOS WFO of the trained policy: same fixed policy, OOS chopped into
#       3 chronological folds. Manual fold-loop (rather than wfo.walk_forward)
#       because there's no hyperparameter optimisation to run — we're just
#       asking "does the edge decay across OOS?".

def _load_sb3(algo: str, path: Path):
    """Lazy SB3 import; load a saved policy."""
    from stable_baselines3 import DQN, PPO
    cls = {"PPO": PPO, "DQN": DQN}[algo]
    return cls.load(str(path), device=DEVICE)


def _best_or_latest_path(algo, g, tf, asset, seed) -> Path | None:
    """Prefer the best-by-val checkpoint; fall back to the most recent step file."""
    prefix = checkpoint_prefix(algo, g, tf, asset, seed)
    best = MODEL_DIR / f"{prefix}_best" / "best_model.zip"
    if best.exists():
        return best
    ckpt, _ = latest_checkpoint(MODEL_DIR, algo, g, tf, asset, seed)
    return ckpt


def wfo_oos_folds(df: pd.DataFrame, params, signals: np.ndarray,
                  n_folds: int = 3) -> tuple[pd.DataFrame, pd.Series]:
    """Backtest the policy on n_folds consecutive chronological slices of df."""
    n = len(df)
    if n == 0:
        return pd.DataFrame(), pd.Series(dtype=float)
    fold_size = n // n_folds
    rows, equity_chunks = [], []
    for k in range(n_folds):
        lo = k * fold_size
        hi = n if k == n_folds - 1 else (k + 1) * fold_size
        sub_df = df.iloc[lo:hi]
        sub_sig = signals[lo:hi]
        strat = DeepRLTradingStrategy(params=params, signal_array=sub_sig)
        res = Backtester(strat).run(sub_df)
        m = compute_metrics(res)
        rows.append({"fold": k, "start": sub_df.index.min(),
                     "end": sub_df.index.max(), "bars": len(sub_df), **m})
        if not res.equity.empty:
            equity_chunks.append(res.equity)
    eq = pd.concat(equity_chunks) if equity_chunks else pd.Series(dtype=float)
    return pd.DataFrame(rows), eq


oos_wfo_rows: list[dict] = []
oos_wfo_curves: dict[tuple, pd.Series] = {}
for algo, g, tf, asset, seed in cell_seeds():
    if seed != 0:
        continue  # one seed's WFO per cell — §12 covers seed dispersion
    path = _best_or_latest_path(algo, g, tf, asset, seed)
    if path is None:
        continue
    ds = spark_grid[g][tf][asset]
    df_clean = clean_split(ds.load(), g); ds.unload()
    oos = slice_by_split(df_clean, g, "oos")
    if oos.empty:
        continue
    params = params_for_cell(g)
    model = _load_sb3(algo, path)
    signals = evaluate_policy_to_signals(model, oos, params)
    folds, eq = wfo_oos_folds(oos, params, signals, n_folds=3)
    for _, r in folds.iterrows():
        oos_wfo_rows.append({"algo": algo, "group": g, "tf": tf, "asset": asset,
                             "seed": seed, **r.to_dict()})
    if not eq.empty:
        oos_wfo_curves[(algo, g, tf, asset)] = eq
    del model
    gc.collect()

oos_wfo_df = pd.DataFrame(oos_wfo_rows)
if oos_wfo_df.empty:
    print("No trained checkpoints found — run §9 first, then re-run §10.")
else:
    cols = ["algo", "group", "tf", "asset", "fold", "bars", "num_trades",
            "total_pnl", "win_rate", "profit_factor", "sharpe_per_trade",
            "max_drawdown"]
    display(oos_wfo_df[cols])

    # Compact dashboard — fold metrics + stitched equity for the best cell.
    best_row = oos_wfo_df.groupby(["algo", "group", "tf", "asset"])\
        ["sharpe_per_trade"].mean().sort_values(ascending=False).head(1)
    if not best_row.empty:
        (algo_b, g_b, tf_b, a_b) = best_row.index[0]
        sub = oos_wfo_df.query("algo==@algo_b and group==@g_b and tf==@tf_b and asset==@a_b")
        eq = oos_wfo_curves.get((algo_b, g_b, tf_b, a_b))
        fig, ax = plt.subplots(1, 2, figsize=(15, 4))
        ax[0].bar(sub["fold"].astype(str), sub["sharpe_per_trade"], color="steelblue")
        ax[0].axhline(0, color="grey", ls="--")
        ax[0].set_title(f"OOS Sharpe-per-trade by fold — {algo_b} {g_b}/{tf_b}/{a_b}")
        ax[0].grid(alpha=.3)
        if eq is not None and not eq.empty:
            ax[1].plot(eq.index, eq.values, color="darkgreen", lw=.9)
            ax[1].set_title("Stitched OOS equity (3 folds)")
            ax[1].grid(alpha=.3)
        fig.tight_layout(); plt.show()


## §11. Full backtest — every `(group, tf, asset)` cell

For each `(algo, group, tf, asset)` cell:

1. Pick the best checkpoint across seeds (max mean-validation-reward).
2. `evaluate_policy_to_signals(model, full_df, deterministic=True)` — *full
   df* meaning train + val + oos concatenated, so the per-fold equity curve
   is comparable to the other strategies' baseline backtests.
3. Wrap with `DeepRLTradingStrategy(signal_array=..., params=...)` and run
   through `Backtester`.
4. Plot `plot_backtest_dashboard(...)` per cell — equity curve, trades,
   drawdowns, metrics panel.
5. Build a single tidy `metrics_df` keyed by
   `(algo, group, tf, asset, seed)`. Pivot to a wide table for inclusion
   in the strategy-comparison dashboard (§13).

Note on cost double-counting: the env already applies `tx_cost_bps` inside
the reward, but the `Backtester` does **not** apply commissions/slippage by
default (`slippage_points=0.0`). They are not double-counted because the
Backtester uses the policy's signal-replay path, not the env's reward path.
This is called out in the §11 markdown summary.


In [ ]:
# §11 — Full backtest of every checkpointed (algo, group, tf, asset, seed).
#       Backtester reads the policy's signal_array (env-side tx cost stays inside
#       the env reward; Backtester uses slippage=0.0 — no double-counting).

backtest_results: dict[tuple, object] = {}  # cache one BacktestResult per cell
metrics_rows: list[dict] = []

for algo, g, tf, asset, seed in cell_seeds():
    path = _best_or_latest_path(algo, g, tf, asset, seed)
    if path is None:
        continue
    params = params_for_cell(g)
    ds = spark_grid[g][tf][asset]
    df_clean = clean_split(ds.load(), g); ds.unload()
    model = _load_sb3(algo, path)
    signals = evaluate_policy_to_signals(model, df_clean, params)
    res = Backtester(
        DeepRLTradingStrategy(params=params, signal_array=signals),
        slippage_points=0.0,
    ).run(df_clean)
    m = compute_metrics(res)
    metrics_rows.append({"algo": algo, "group": g, "tf": tf, "asset": asset,
                         "seed": seed, **m})
    backtest_results[(algo, g, tf, asset, seed)] = res
    del model
    gc.collect()

metrics_df = pd.DataFrame(metrics_rows)
if metrics_df.empty:
    print("No metrics — train (§9) at least one cell first.")
else:
    display(metrics_df.sort_values("sharpe_per_trade", ascending=False).head(20))
    # Wide pivot (algo × asset/seed) for inclusion in the comparison dashboard.
    wide = metrics_df.pivot_table(
        index=["group", "tf", "asset"], columns="algo",
        values=["sharpe_per_trade", "profit_factor", "total_pnl"]
    )
    display(wide)

    # Showcase dashboard: top cell by sharpe_per_trade.
    best = metrics_df.sort_values("sharpe_per_trade", ascending=False).iloc[0]
    key = (best["algo"], best["group"], best["tf"], best["asset"], int(best["seed"]))
    res = backtest_results[key]
    fig = plot_backtest_dashboard(
        res, title=f"{best['algo']} {best['group']}/{best['tf']}/{best['asset']}  seed={best['seed']}"
    )
    plt.show()


## §12. Overfitting and robustness checks

The classical robustness suite from [[06_Robustness_Testing]], adapted to RL:

### 12.1 Seed sensitivity — RL-specific

The single most important RL robustness check. For each
`(algo, group, tf, asset)` cell:

- Plot OOS equity curves of all `n_seeds` seeds on the same axes.
- Report **mean ± std** of OOS Sharpe / Profit Factor across seeds.
- If std/mean > 1 (high relative variance), flag the cell as "unstable —
  the policy is at the mercy of seed luck".

### 12.2 Block bootstrap on OOS trades

`block_bootstrap_trades(trades_df, block_size_bars=...)` per cell. Reports
the bootstrap distribution of OOS Sharpe and `P(Sharpe > 0)`.

### 12.3 Sub-period analysis

`subperiod_analysis(trades_df, freq="YE")` per cell. Confirms the policy
isn't carried by a single year of OOS performance.

### 12.4 Parameter sensitivity — *post-training*

`parameter_sensitivity(...)` over a small grid of *inference-time* knobs:

- `deterministic ∈ {True, False}` — does stochastic policy roll-out
  meaningfully change outcomes?
- `tx_cost_bps × {0.5, 1.0, 2.0}` — replay through the Backtester with a
  cost overlay (the eval-time cost can differ from the train-time cost) to
  measure cost sensitivity.

(Note: classical hyperparameter sensitivity requires re-training the policy
per point — already covered by §9.1's grid.)

### 12.5 Synthetic asset null hypothesis

Same H₀-overfitted-hypothesis-test framing as notebook 15 §7: train the
policy on a Geometric-Brownian-Motion synthetic asset and verify that OOS
performance drops to ~0 (no real edge to learn from). If the policy still
"works" on synthetic data, it's overfitting to noise.

This is the most diagnostic single check for an RL policy. Done as a single
quick training run per algo on a synthetic of matched volatility, **not** as
a full sweep.


In [ ]:
# §12 — Robustness suite (1 cell, 5 sub-checks).
if metrics_df.empty:
    print("Robustness suite skipped — train (§9) at least one cell first.")
else:
    # --- 12.1 Seed sensitivity --------------------------------------------
    by_cell = metrics_df.groupby(["algo", "group", "tf", "asset"])
    seed_rows = []
    for keys, grp in by_cell:
        s = grp["sharpe_per_trade"]; pf = grp["profit_factor"]
        seed_rows.append({
            "algo": keys[0], "group": keys[1], "tf": keys[2], "asset": keys[3],
            "n_seeds": len(grp),
            "sharpe_mean": s.mean(), "sharpe_std": s.std(),
            "pf_mean": pf.mean(), "pf_std": pf.std(),
            "rel_std_sharpe": (s.std() / max(abs(s.mean()), 1e-9)),
            "unstable_flag": (s.std() / max(abs(s.mean()), 1e-9)) > 1.0,
        })
    seed_df = pd.DataFrame(seed_rows)
    print("§12.1 — seed sensitivity (mean ± std across seeds):")
    display(seed_df.sort_values("sharpe_mean", ascending=False))

    # OOS equity overlay for the top cell
    top = seed_df.sort_values("sharpe_mean", ascending=False).iloc[0]
    fig, ax = plt.subplots(figsize=(13, 5))
    for seed in range(RL_CONFIG["n_seeds"]):
        path = _best_or_latest_path(top["algo"], top["group"], top["tf"], top["asset"], seed)
        if path is None:
            continue
        params = params_for_cell(top["group"])
        ds = spark_grid[top["group"]][top["tf"]][top["asset"]]
        df_clean = clean_split(ds.load(), top["group"]); ds.unload()
        oos = slice_by_split(df_clean, top["group"], "oos")
        model = _load_sb3(top["algo"], path)
        sig = evaluate_policy_to_signals(model, oos, params)
        res = Backtester(DeepRLTradingStrategy(params=params, signal_array=sig)).run(oos)
        if not res.equity.empty:
            ax.plot(res.equity.index, res.equity.values, lw=.9, label=f"seed={seed}")
        del model; gc.collect()
    ax.set_title(f"OOS equity per seed — {top['algo']} {top['group']}/{top['tf']}/{top['asset']}")
    ax.legend(); ax.grid(alpha=.3); plt.show()

    # --- 12.2 Block bootstrap (showcase cell, all seeds combined) ---------
    print("\n§12.2 — block bootstrap on OOS trades (showcase cell):")
    trades_all = []
    for seed in range(RL_CONFIG["n_seeds"]):
        k = (top["algo"], top["group"], top["tf"], top["asset"], seed)
        r = backtest_results.get(k)
        if r is not None and not r.trades.empty:
            trades_all.append(r.trades)
    if trades_all:
        all_trades = pd.concat(trades_all, ignore_index=True)
        bb = block_bootstrap_trades(all_trades, n_runs=500, seed=GLOBAL_SEED)
        if not bb.empty:
            final = bb.iloc[-1]
            print(f"  trades={len(all_trades):>5d}  "
                  f"P(final>0)={float((final > 0).mean()):.3f}  "
                  f"median_final={float(final.median()):.1f}  "
                  f"q05={float(final.quantile(.05)):.1f}  q95={float(final.quantile(.95)):.1f}")

    # --- 12.3 Sub-period analysis (yearly) --------------------------------
    print("\n§12.3 — yearly sub-period metrics (showcase cell, seed=0):")
    r0 = backtest_results.get((top["algo"], top["group"], top["tf"], top["asset"], 0))
    if r0 is not None and not r0.trades.empty:
        display(subperiod_analysis(r0.trades, freq="YE").round(3))

    # --- 12.4 Inference-time parameter sensitivity ------------------------
    print("\n§12.4 — inference-time sensitivity (deterministic flag, tx_cost overlay):")
    params = params_for_cell(top["group"])
    ds = spark_grid[top["group"]][top["tf"]][top["asset"]]
    df_clean = clean_split(ds.load(), top["group"]); ds.unload()
    full_df = df_clean
    sens_rows = []
    for seed in range(RL_CONFIG["n_seeds"]):
        path = _best_or_latest_path(top["algo"], top["group"], top["tf"], top["asset"], seed)
        if path is None:
            continue
        model = _load_sb3(top["algo"], path)
        # deterministic on/off — call the env loop manually for the off variant.
        for det in (True, False):
            env_eval = TradingEnv(full_df, params, mode="eval")
            obs, _ = env_eval.reset()
            sig = np.zeros(len(full_df), dtype=int)
            while True:
                t = env_eval.t
                a, _ = model.predict(obs, deterministic=det)
                sig[t] = ACTION_TO_SIGNAL[int(a)]
                obs, _r, term, trunc, _i = env_eval.step(int(a))
                if term or trunc:
                    break
            # tx-cost overlay {0.5, 1.0, 2.0} × baseline; replay through Backtester
            for mult in (0.5, 1.0, 2.0):
                p_overlay = replace(params, tx_cost_bps=params.tx_cost_bps * mult)
                res = Backtester(DeepRLTradingStrategy(
                    params=p_overlay, signal_array=sig)).run(full_df)
                m = compute_metrics(res)
                sens_rows.append({"seed": seed, "deterministic": det,
                                  "tx_cost_mult": mult,
                                  "sharpe": m["sharpe_per_trade"],
                                  "total_pnl": m["total_pnl"],
                                  "num_trades": m["num_trades"]})
        del model; gc.collect()
    sens_df = pd.DataFrame(sens_rows)
    if not sens_df.empty:
        display(sens_df.pivot_table(
            index=["seed", "deterministic"], columns="tx_cost_mult",
            values=["sharpe", "total_pnl", "num_trades"]).round(3))

    # --- 12.5 Synthetic-asset null hypothesis -----------------------------
    print("\n§12.5 — synthetic-asset null hypothesis (function defined; opt-in to run):")
    def synthetic_null(algo: str, group: str = "forex", n_bars: int = 20_000,
                       seed: int = GLOBAL_SEED, total_timesteps: int = 100_000) -> dict:
        """Train one quick policy on a GBM synthetic of matched volatility.
        Returns OOS metrics; OOS Sharpe ≈ 0 ⇒ the model isn't overfitting noise."""
        rng = np.random.default_rng(seed)
        vol = 0.01
        idx = pd.date_range("2020-01-01", periods=n_bars, freq="1h")
        rets = rng.normal(0.0, vol, n_bars)
        close = 100.0 * np.exp(np.cumsum(rets))
        high = close * (1 + np.abs(rng.normal(0, vol / 2, n_bars)))
        low  = close * (1 - np.abs(rng.normal(0, vol / 2, n_bars)))
        df = pd.DataFrame({"open": close, "high": high, "low": low,
                           "close": close,
                           "tick_vol": rng.integers(100, 200, n_bars)}, index=idx)
        cut = int(n_bars * 0.7)
        params = params_for_cell(group)
        sub_dir = MODEL_DIR / f"_synth_{algo}_{seed}"
        info = train_one_seed(
            df.iloc[:cut], df.iloc[cut:int(n_bars * 0.85)], params,
            algo=algo, group="synth", tf="1h", asset="GBM", seed=seed,
            model_dir=sub_dir, total_timesteps=total_timesteps,
            n_envs=1, save_freq=total_timesteps // 2,
            eval_freq=total_timesteps // 4, device=DEVICE,
            tensorboard_log=str(TB_LOG_DIR / "synth"), subproc=False,
        )
        model = _load_sb3(algo, Path(info["final_path"]))
        oos = df.iloc[cut:]
        sig = evaluate_policy_to_signals(model, oos, params)
        res = Backtester(DeepRLTradingStrategy(params=params, signal_array=sig)).run(oos)
        return compute_metrics(res)

    print("  call synthetic_null('PPO') to run (writes under "
          f"{MODEL_DIR}/_synth_PPO_*); a healthy result has OOS Sharpe ≈ 0.")


## §13. Comparison with benchmarks

Side-by-side OOS performance of the RL policy vs the established repo
strategies, evaluated on the **same** OOS window and the **same** asset cells.

### Benchmarks pulled from the existing repo

- **Strategy 01 — SMA Crossover ATR Risk.** The unoptimised baseline; the
  bar an RL policy must clear to be interesting at all.
- **Strategy 10 — HMM Regime Filter (GaussianMixture proxy).** The repo's
  other ML-based strategy. Comparing against #10 separates the "learned vs
  hand-coded" axis (RL vs SMA) from the "ML vs ML" axis (RL vs HMM).
- **Buy-and-hold.** Trivial but essential — if RL doesn't beat passive on
  the OOS window, no further analysis is needed.

### Mechanics

- Use `source.comparison.STRATEGY_REGISTRY` to pull the baseline configs.
- Run #01 and #10 through `Backtester` on the same OOS slices the RL
  policies are evaluated on (cache lookup if the `comparison/.cache/` hit
  is fresh enough; otherwise rerun).
- Build a single comparison table (rows = strategies, columns = OOS metrics
  per cell).
- Plot `plot_strategy_equity_overlay(...)` for the top-N cells.

### Acceptance criteria (pre-defined, copied from strategy doc)

The RL agent **passes** the comparison only if:

- **Mean OOS Sharpe across seeds beats both SMA #01 and HMM #10** on at
  least 2 of the 4 representative cells (forex-1h-EURUSD, forex-4h-EURUSD,
  b3-30min-WIN, b3-1h-WIN), **and**
- **Worst-seed Sharpe is positive** on those cells.

Otherwise: report negative result in §14 and leave Status = "Backtested —
did not beat baseline".


In [ ]:
# §13 — Comparison with benchmark strategies (SMA #01, HMM #10, buy-and-hold).
#       Run them on the SAME OOS slice each cell uses, then compare metrics.
BASELINE_NAMES = ("SMA Crossover", "HMM Regime Filter")
benchmarks = {r.name: r for r in STRATEGY_REGISTRY if r.name in BASELINE_NAMES}
print(f"Benchmark registry hits: {list(benchmarks)}")


def buy_and_hold_metrics(df: pd.DataFrame) -> dict:
    """Trivial passive baseline (close-to-close PnL)."""
    if df.empty:
        return {"num_trades": 0, "total_pnl": 0.0,
                "sharpe_per_trade": 0.0, "max_drawdown": 0.0}
    px = df["close"].astype(float).values
    eq = pd.Series(px - px[0], index=df.index)
    return {"num_trades": 1, "total_pnl": float(eq.iloc[-1]),
            "sharpe_per_trade": float(eq.diff().mean()
                                       / (eq.diff().std() + 1e-9)),
            "max_drawdown": float((eq - eq.cummax()).min())}


def _run_benchmark_on_cell(reg, group, tf, asset, oos_df) -> dict:
    """Build the strategy from registry, run Backtester on the OOS slice."""
    params = make_params_for_group(reg, group)
    strat = reg.strategy_cls(params)
    res = Backtester(strat).run(oos_df)
    return compute_metrics(res)


comparison_rows = []
if metrics_df.empty:
    print("§13 needs §11 metrics — train at least one cell first.")
else:
    for g, tfs in GROUP_TIMEFRAMES.items():
        for tf in tfs:
            for asset in ASSETS[g]:
                if asset not in spark_grid.get(g, {}).get(tf, {}):
                    continue
                ds = spark_grid[g][tf][asset]
                df_clean = clean_split(ds.load(), g); ds.unload()
                oos = slice_by_split(df_clean, g, "oos")
                if oos.empty:
                    continue

                # RL benchmarks (mean across seeds, per algo)
                for algo in RL_CONFIG["algorithms"]:
                    sub = metrics_df.query(
                        "algo==@algo and group==@g and tf==@tf and asset==@asset")
                    if not sub.empty:
                        comparison_rows.append({
                            "strategy": f"RL-{algo}",
                            "group": g, "tf": tf, "asset": asset,
                            "total_pnl": sub["total_pnl"].mean(),
                            "sharpe_per_trade": sub["sharpe_per_trade"].mean(),
                            "profit_factor": sub["profit_factor"].mean(),
                            "win_rate": sub["win_rate"].mean(),
                            "max_drawdown": sub["max_drawdown"].mean(),
                            "n_seeds": len(sub),
                        })

                # Each registered baseline (only if its registry says (g, tf) is supported)
                for name, reg in benchmarks.items():
                    if tf not in reg.group_timeframes.get(g, []):
                        continue
                    try:
                        m = _run_benchmark_on_cell(reg, g, tf, asset, oos)
                    except Exception as e:
                        print(f"  [{name}] {g}/{tf}/{asset} failed: {e!r}")
                        continue
                    comparison_rows.append({
                        "strategy": name, "group": g, "tf": tf, "asset": asset,
                        **{k: m.get(k) for k in
                           ("total_pnl", "sharpe_per_trade", "profit_factor",
                            "win_rate", "max_drawdown")},
                        "n_seeds": 1,
                    })

                # Buy-and-hold
                bh = buy_and_hold_metrics(oos)
                comparison_rows.append({"strategy": "Buy-and-Hold",
                                        "group": g, "tf": tf, "asset": asset,
                                        **bh, "n_seeds": 1,
                                        "profit_factor": np.nan, "win_rate": np.nan})

    comp_df = pd.DataFrame(comparison_rows)
    display(comp_df.sort_values(["group", "tf", "asset", "sharpe_per_trade"],
                                ascending=[True, True, True, False]))

    # Pivot for an at-a-glance Sharpe board
    sharpe_pivot = comp_df.pivot_table(
        index=["group", "tf", "asset"], columns="strategy",
        values="sharpe_per_trade").round(3)
    print("\nOOS Sharpe-per-trade (rows = cells, cols = strategy):")
    display(sharpe_pivot)

    # Acceptance: RL mean > both SMA and HMM on >=2 of the representative cells,
    # AND worst-seed Sharpe > 0 on those cells.
    representative = [("forex", "1h", "EURUSD"), ("forex", "4h", "EURUSD"),
                      ("b3", "30min", "WIN"), ("b3", "1h", "WIN")]
    passed = []
    for cell in representative:
        sub = comp_df[(comp_df["group"] == cell[0]) &
                      (comp_df["tf"]    == cell[1]) &
                      (comp_df["asset"] == cell[2])]
        if sub.empty:
            continue
        rl_best = sub[sub["strategy"].str.startswith("RL-")]["sharpe_per_trade"].max()
        sma = sub.loc[sub["strategy"] == "SMA Crossover", "sharpe_per_trade"]
        hmm = sub.loc[sub["strategy"] == "HMM Regime Filter", "sharpe_per_trade"]
        sma_v = float(sma.iloc[0]) if not sma.empty else float("-inf")
        hmm_v = float(hmm.iloc[0]) if not hmm.empty else float("-inf")
        beat = (rl_best > sma_v) and (rl_best > hmm_v)

        # worst-seed Sharpe across both algos for this cell
        worst = metrics_df.query(
            "group==@cell[0] and tf==@cell[1] and asset==@cell[2]"
        )["sharpe_per_trade"]
        worst_v = float(worst.min()) if not worst.empty else float("-inf")
        passed.append({"cell": cell, "rl_best_sharpe": rl_best,
                       "sma_sharpe": sma_v, "hmm_sharpe": hmm_v,
                       "worst_seed_sharpe": worst_v,
                       "beats_baselines": beat, "worst_seed_positive": worst_v > 0,
                       "passes": beat and (worst_v > 0)})
    accept = pd.DataFrame(passed)
    print("\nAcceptance criteria summary:")
    display(accept)
    n_pass = int(accept["passes"].sum()) if not accept.empty else 0
    print(f"\nCells passing: {n_pass} / {len(accept)}.  "
          + ("→ ACCEPT — write up positively in §14."
             if n_pass >= 2 else "→ REJECT — write up the negative result in §14."))


## §14. Findings and next steps

Populated post-run. The cell will summarise:

- Whether the acceptance criteria from §13 were met.
- Which algorithm/cell combinations were stable across seeds vs which were
  seed-luck.
- The §12.5 synthetic-null result (the single most credible robustness
  signal).
- A ranked list of v2 follow-ups, draft of which already lives in the
  strategy doc's *Known Weaknesses* section:
    - Enable `use_differential_sharpe` and A/B against per-step PnL reward.
    - Multi-asset env (cycle assets per episode).
    - HMM-state pre-conditioning (ensemble #10 + #16).
    - LSTM policy via `sb3_contrib.RecurrentPPO`.
    - "Policy mode" Backtester extension (`use_flat_close_signal`).
    - Continuous position sizing (`use_continuous_action`) — requires
      `PortfolioBacktester` to be on `main`.
- Open issue numbers (TBD on PR open) for each follow-up.

Notebook commits **unexecuted**, per repo convention.


In [ ]:
# §14 — Findings table. Populated when §11/§12/§13 have run; the markdown
#       narrative above should reference these numbers verbatim.
findings_rows: list[dict] = []
if not metrics_df.empty:
    # Mean OOS metrics per (algo, group, tf, asset)
    oos_summary = (
        metrics_df
        .groupby(["algo", "group", "tf", "asset"])
        .agg(sharpe_mean=("sharpe_per_trade", "mean"),
             sharpe_std =("sharpe_per_trade", "std"),
             pnl_mean   =("total_pnl",        "mean"),
             pf_mean    =("profit_factor",    "mean"),
             dd_mean    =("max_drawdown",     "mean"),
             n_seeds    =("seed",             "count"))
        .reset_index()
    )
    findings_rows = oos_summary.to_dict(orient="records")
    display(oos_summary.sort_values("sharpe_mean", ascending=False))

# Cleanly shut down Spark — final step.
if spark is not None:
    spark.stop()
    print("Spark session stopped. Notebook commits unexecuted (per repo convention).")
